# 03. Legal KG Construction — 교통법령 Knowledge Graph 구축
**산출물**: Neo4j KG (약 60 노드 (v3 확장 포함)) + kg_snapshot.json + kg_validation.json

**의존**: 01_data_collection.py → core_articles.json 등

**스키마**:
```
(HazardousBehavior)-[:VIOLATES]->(LegalArticle)
(LegalArticle)-[:RELATED_TO]->(LegalArticle)
(SeverityLevel)-[:PENALIZED_BY]->(Penalty)
(AggravatingFactor)-[:AGGRAVATES]->(SeverityLevel)
(LegalArticle)-[:DEFINED_BY]->(LegalArticle)      ★ v3 추가
(SeverityLevel)-[:ESCALATES_TO]->(SeverityLevel)   ★ v3 추가
```

**v3 확장**: 시행령 별표, 교통안전법, 화물차법, 교통사고특례법 조항 추가 → 약 60 노드 (v3 확장 포함)

In [1]:
import os, json, pathlib
from dotenv import load_dotenv
load_dotenv()

from config import (load_json, save_json, JSON_DIR, PENALTY_DIR, KG_DIR, RESULTS_DIR,
                    SEVERITY_TABLE, AGGRAVATING_FACTORS)

CORE_ARTICLES = load_json(JSON_DIR / 'core_articles.json')
# ★ VIOLATION_MAPPING은 JSON에서 로드 (config.py는 10개, JSON은 13개)
# 01_data_collection에서 추가한 dtg_violation, education, overload 포함
VIOLATION_MAPPING = load_json(JSON_DIR / 'violation_article_mapping.json')
print(f'로드: 조문 {len(CORE_ARTICLES)}개, 위반유형 {len(VIOLATION_MAPPING)}개, '
      f'심각도 {len(SEVERITY_TABLE)}단계, 가중 {len(AGGRAVATING_FACTORS)}개')

로드: 조문 15개, 위반유형 13개, 심각도 6단계, 가중 2개


## 1. Neo4j 연결

In [2]:
from neo4j import GraphDatabase

NEO4J_URI  = os.getenv('NEO4J_URI', '')
NEO4J_USER = os.getenv('NEO4J_USER', '')
NEO4J_PW   = os.getenv('NEO4J_PW', '')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PW))

def run_q(query, params=None):
    with driver.session() as s:
        return [dict(r) for r in s.run(query, params or {})]

r = run_q("RETURN 'Connected' AS msg")
print(f'✅ Neo4j: {r[0]["msg"]}')

✅ Neo4j: Connected


## 2. DB 초기화

In [3]:
run_q('MATCH (n) DETACH DELETE n')
print('✅ KG 초기화')

✅ KG 초기화


## 3~7. 노드 생성

In [4]:
# ── HazardousBehavior (13개 — config 10 + JSON 추가 3) ──
for vtype, info in VIOLATION_MAPPING.items():
    run_q("""CREATE (h:HazardousBehavior {
        name:$name, name_kr:$kr, threshold:$th, has_severity:$hs})""",
        {'name':vtype,'kr':info['name_kr'],'th':info['etas_threshold'],
         'hs':info.get('has_severity_levels',False)})
print(f'✅ HazardousBehavior: {run_q("MATCH (h:HazardousBehavior) RETURN count(h) AS c")[0]["c"]}개')

# ── LegalArticle (15개 — core_articles.json) ──
for art in CORE_ARTICLES:
    run_q("""CREATE (a:LegalArticle {
        id:$id, law:$law, jo_num:$jo, title:$title, content:$content,
        type:$type, violation_type:$vt, source:$src})""",
        {'id':art['id'],'law':art['law'],'jo':art['jo_num'],'title':art['title'],
         'content':art.get('content',''),'type':art['type'],
         'vt':art.get('violation_type',''),'src':art.get('source','')})
print(f'✅ LegalArticle: {run_q("MATCH (a:LegalArticle) RETURN count(a) AS c")[0]["c"]}개')

# ── SeverityLevel (6개) ──
for lk, info in SEVERITY_TABLE.items():
    run_q("""CREATE (s:SeverityLevel {
        id:$id, label:$label, speed_over_min:$smin, speed_over_max:$smax,
        description:$desc, demerit_points:$dp, legal_basis:$basis, criminal:$crim})""",
        {'id':lk,'label':info['label'],'smin':info['speed_over_min'],
         'smax':info.get('speed_over_max',999),'desc':info['description'],
         'dp':info.get('demerit_points') if info.get('demerit_points') is not None else -1,
         'basis':info['legal_basis'],'crim':info['criminal']})
print(f'✅ SeverityLevel: {run_q("MATCH (s:SeverityLevel) RETURN count(s) AS c")[0]["c"]}개')

# ── Penalty (13개) ──
penalties = [
    {'id':'P_FINE_3','type':'범칙금','value':'3만원','cond':'20km/h이하','criminal':False},
    {'id':'P_FINE_6','type':'범칙금','value':'6만원','cond':'20~40km/h','criminal':False},
    {'id':'P_FINE_9','type':'범칙금','value':'9만원','cond':'40~60km/h','criminal':False},
    {'id':'P_FINE_12','type':'범칙금','value':'12만원','cond':'60~80km/h','criminal':False},
    {'id':'P_DEMERIT_15','type':'벌점','value':'15점','cond':'20~40km/h','criminal':False},
    {'id':'P_DEMERIT_30','type':'벌점','value':'30점','cond':'40~60km/h','criminal':False},
    {'id':'P_DEMERIT_60','type':'벌점','value':'60점','cond':'60~80km/h(면허정지)','criminal':False},
    {'id':'P_DEMERIT_80','type':'벌점','value':'80점','cond':'80~100km/h','criminal':False},
    {'id':'P_CRIMINAL_30','type':'벌금','value':'30만원이하','cond':'80~100km/h','criminal':True},
    {'id':'P_CRIMINAL_100','type':'벌금','value':'100만원이하','cond':'100km/h초과','criminal':True},
    {'id':'P_CRIMINAL_500','type':'징역/벌금','value':'1년이하 징역 또는 500만원이하',
     'cond':'100km/h 3회이상','criminal':True},
    {'id':'P_LICENSE_SUSPEND','type':'면허정지','value':'면허정지','cond':'벌점 40점이상','criminal':False},
    {'id':'P_LICENSE_CANCEL','type':'면허취소','value':'면허취소','cond':'개별기준','criminal':False},
]
for p in penalties:
    run_q("CREATE (p:Penalty {id:$id,type:$type,value:$value,condition:$cond,criminal:$criminal})", p)
print(f'✅ Penalty: {run_q("MATCH (p:Penalty) RETURN count(p) AS c")[0]["c"]}개')

# ── AggravatingFactor (2개) ──
for ak, info in AGGRAVATING_FACTORS.items():
    run_q("CREATE (g:AggravatingFactor {id:$id, name:$name, legal_basis:$basis})",
          {'id':ak,'name':info['name'],'basis':info.get('legal_basis',info.get('escalation',''))})
print(f'✅ AggravatingFactor: {run_q("MATCH (g:AggravatingFactor) RETURN count(g) AS c")[0]["c"]}개')

# ═══════════════════════════════════════════════════════════
# ★ v3: KG 확장 — 추가 법령 조항 노드
# ═══════════════════════════════════════════════════════════

# ── 시행령 별표 노드 ──
enforcement_articles = [
    {'id':'RTAE_T7','law':'도로교통법 시행령','jo_num':'별표 7',
     'title':'운전면허 행정처분 기준(벌점)','type':'penalty_table',
     'content':'과속 20km/h이하: 벌점 없음, 20~40: 15점, 40~60: 30점, 60~80: 60점'},
    {'id':'RTAE_T8','law':'도로교통법 시행령','jo_num':'별표 8',
     'title':'범칙금액(과속)','type':'penalty_table',
     'content':'승합·화물 과속 20km/h이하: 3만원, 20~40: 7만원, 40~60: 10만원, 60~80: 13만원'},
    {'id':'RTAE_T10','law':'도로교통법 시행령','jo_num':'별표 10',
     'title':'어린이보호구역 범칙금','type':'penalty_table',
     'content':'어린이보호구역 과속 범칙금 가중 기준'},
    {'id':'RTAR_T28','law':'도로교통법 시행규칙','jo_num':'별표 28',
     'title':'운전면허 취소·정지처분 기준(벌점)','type':'penalty_table',
     'content':'벌점에 의한 면허 취소·정지 기준: 누산점수 40점이상 면허정지, 개별기준 면허취소'},
]

# ── 교통안전법 조항 ──
safety_articles = [
    {'id':'TSA_54','law':'교통안전법','jo_num':'제54조',
     'title':'교통안전담당자 선임 의무','type':'definition',
     'content':'교통수단 운영자는 교통안전에 관한 업무를 담당할 교통안전담당자를 선임하여야 한다'},
    # ★ TSA_54_2는 core_articles.json에서 이미 생성됨 → 중복 제거
    # (제54조의2 = 교통안전담당자 지정, DTG 장착은 제55조)
    {'id':'TSA_55_3','law':'교통안전법','jo_num':'제55조제3항',
     'title':'운행기록 분석 결과 제공','type':'procedure',
     'content':'국토교통부장관은 분석 결과를 운수업체에 제공할 수 있다'},
    {'id':'TSA_55_4','law':'교통안전법','jo_num':'제55조제4항',
     'title':'운행기록 목적외 사용 금지','type':'restriction',
     'content':'운행기록 분석 결과는 교통안전 점검 및 안전관리 목적으로만 사용'},
]

# ── 화물자동차 운수사업법 ──
# ★ TTBA_59는 core_articles.json에서 이미 생성됨 → FTA_59 중복 제거
freight_articles = []

# ── 교통사고처리특례법 ──
special_articles = [
    {'id':'TASA_3','law':'교통사고처리특례법','jo_num':'제3조',
     'title':'처벌의 특례','type':'penalty',
     'content':'업무상과실 또는 중대한 과실로 교통사고를 일으킨 경우 처벌 특례'},
    {'id':'TASA_4','law':'교통사고처리특례법','jo_num':'제4조',
     'title':'보험 등에 가입된 경우의 특례','type':'penalty',
     'content':'종합보험 가입 시 공소 제기 특례'},
]

# ── 특정범죄 가중처벌법 ──
aggravated_articles = [
    {'id':'SPCA_5_3','law':'특정범죄 가중처벌법','jo_num':'제5조의3',
     'title':'도주차량 운전자 가중처벌','type':'penalty',
     'content':'사고 후 도주 시 가중처벌'},
    {'id':'SPCA_5_11','law':'특정범죄 가중처벌법','jo_num':'제5조의11',
     'title':'위험운전 치상·치사','type':'penalty',
     'content':'음주 또는 약물 영향 하 위험운전으로 사상 시 가중처벌'},
]

# 차종별 범칭금 Penalty 노드
fine_penalties = [
    {'id':'FINE_L1_FREIGHT','value':'화물차 과속 20km/h이하 범칙금 3만원','level':'level_1','vehicle':'freight'},
    {'id':'FINE_L2_FREIGHT','value':'화물차 과속 20~40km/h 범칙금 7만원','level':'level_2','vehicle':'freight'},
    {'id':'FINE_L3_FREIGHT','value':'화물차 과속 40~60km/h 범칙금 10만원','level':'level_3','vehicle':'freight'},
    {'id':'FINE_L4_FREIGHT','value':'화물차 과속 60~80km/h 범칙금 13만원','level':'level_4','vehicle':'freight'},
]

all_new_articles = enforcement_articles + safety_articles + freight_articles + special_articles + aggravated_articles

for art in all_new_articles:
    run_q("""CREATE (a:LegalArticle {
        id:$id, law:$law, jo_num:$jo, title:$title, type:$type, content:$content})""",
        {'id':art['id'],'law':art['law'],'jo':art['jo_num'],
         'title':art['title'],'type':art['type'],'content':art['content']})

for pen in fine_penalties:
    run_q("""CREATE (p:Penalty {
        id:$id, value:$value, level:$level, vehicle:$vehicle})""",
        pen)

print(f'✅ v3 확장 노드 추가: {len(all_new_articles)} LegalArticle + {len(fine_penalties)} Penalty')

# ★ 심각도 6단계 분류 기준표 (KG 노드 — S3에서 참조)
SEVERITY_GUIDE_TEXT = (
    '과속 심각도 6단계 분류 기준 (초과속도 구간별 법적 처분): '
    'level_1 (경미): 초과속도 0~20km/h 이하, 범칙금만(화물차 3만원), 벌점 없음, 제156조 적용. '
    'level_2 (주의): 초과속도 20~40km/h, 범칙금(화물차 7만원)+벌점 15점, 제156조 적용. '
    'level_3 (경고): 초과속도 40~60km/h, 범칙금(화물차 10만원)+벌점 30점, 제156조 적용. '
    'level_4 (위험): 초과속도 60~80km/h, 범칙금(화물차 13만원)+벌점 60점(면허정지), 제156조 적용. '
    'level_5 (매우위험): 초과속도 80~100km/h, 형사처벌(30만원 이하 벌금 또는 구류)+벌점 80점, 제154조제9호. '
    'level_6 (극히위험): 초과속도 100km/h 초과, 형사처벌(100만원 이하 벌금 또는 구류), 제153조제2항제2호. '
    '3회 이상 100km/h 초과 시 제151조의2 적용(1년이하 징역/500만원이하 벌금).'
)

run_q("""CREATE (a:LegalArticle {
    id:'SGT_SEVERITY_GUIDE', law:'과속 심각도 분류', jo_num:'6단계 기준표',
    title:'과속 심각도 6단계 분류 기준', type:'penalty_table',
    content:$content})""", {'content': SEVERITY_GUIDE_TEXT})
print(f'✅ 심각도 가이드 노드 추가: SGT_SEVERITY_GUIDE')



✅ HazardousBehavior: 13개
✅ LegalArticle: 15개
✅ SeverityLevel: 6개
✅ Penalty: 13개
✅ AggravatingFactor: 2개
✅ v3 확장 노드 추가: 11 LegalArticle + 4 Penalty
✅ 심각도 가이드 노드 추가: SGT_SEVERITY_GUIDE


## 8~12. 엣지 생성

In [5]:
# ── VIOLATES ──
v_cnt = 0
for vtype, info in VIOLATION_MAPPING.items():
    all_arts = info.get('definition_articles',[]) + info.get('penalty_articles',[]) + info.get('aggravation',[])
    for art_id in all_arts:
        run_q("MATCH (h:HazardousBehavior {name:$v}) MATCH (a:LegalArticle {id:$a}) CREATE (h)-[:VIOLATES]->(a)",
              {'v':vtype,'a':art_id})
        v_cnt += 1
print(f'✅ VIOLATES: {v_cnt}개')

# ── CLASSIFIED_AS ──
s_cnt = 0
for vtype, info in VIOLATION_MAPPING.items():
    if info.get('has_severity_levels'):
        for lk in SEVERITY_TABLE:
            run_q("MATCH (h:HazardousBehavior {name:$v}) MATCH (s:SeverityLevel {id:$l}) CREATE (h)-[:CLASSIFIED_AS]->(s)",
                  {'v':vtype,'l':lk})
            s_cnt += 1
print(f'✅ CLASSIFIED_AS: {s_cnt}개')

# ── RELATED_TO ──
related = [
    ('RTA_17','RTA_156','속도위반→벌칙(20만이하)'),
    ('RTA_17','RTA_154','속도위반→벌칙(30만이하)'),
    ('RTA_17','RTA_153','속도위반→벌칙(100만이하)'),
    ('RTA_17','RTA_151_2','속도위반→반복초과속'),
    ('RTA_49','RTA_156','급제동등→벌칙'),
    ('RTA_19','RTA_156','진로변경위반→벌칙'),
    ('RTA_21','RTA_156','앞지르기위반→벌칙'),
    ('RTA_25','RTA_156','교차로통행위반→벌칙'),
    ('RTA_12','RTA_17','어린이보호구역→속도위반가중'),
    # ★ TSA_56 제거 (교통안전체험시설 조문으로 개정됨)
    # DTG 제출 의무는 TSA_55 제2항에 포함. TSA_55→TTBA_11 직접 연결
    ('TSA_55','TTBA_11','DTG 미제출→행정처분'),
    ('TSA_54_2','TSA_55','안전담당자→DTG관리'),
    ('TTBA_59','TTBA_11','교육미이수→행정처분'),
]
for src, tgt, desc in related:
    run_q("MATCH (a1:LegalArticle {id:$s}) MATCH (a2:LegalArticle {id:$t}) CREATE (a1)-[:RELATED_TO {description:$d}]->(a2)",
          {'s':src,'t':tgt,'d':desc})
print(f'✅ RELATED_TO: {len(related)}개')

# ── PENALIZED_BY ──
pen_map = [('level_1',['P_FINE_3']),('level_2',['P_FINE_6','P_DEMERIT_15']),
           ('level_3',['P_FINE_9','P_DEMERIT_30']),('level_4',['P_FINE_12','P_DEMERIT_60']),
           ('level_5',['P_CRIMINAL_30','P_DEMERIT_80']),
           ('level_6',['P_CRIMINAL_100','P_CRIMINAL_500','P_LICENSE_CANCEL'])]
pe = 0
for lv, pids in pen_map:
    for pid in pids:
        run_q("MATCH (s:SeverityLevel {id:$l}) MATCH (p:Penalty {id:$p}) CREATE (s)-[:PENALIZED_BY]->(p)",
              {'l':lv,'p':pid})
        pe += 1
print(f'✅ PENALIZED_BY: {pe}개')

# ═══════════════════════════════════════════════════════════
# ★ v3: 추가 관계
# ═══════════════════════════════════════════════════════════

# DEFINED_BY: 벌칙조문 → 별표 참조
defined_by_edges = [
    ('RTA_156', 'RTAE_T8'),   # 범칙금 → 별표8 범칙금액
    ('RTA_156', 'RTAE_T7'),   # 범칙금 → 별표7 벌점
    ('RTA_12', 'RTAE_T10'),   # 보호구역 → 별표10 가중범칙금
]
for src_id, tgt_id in defined_by_edges:
    run_q("MATCH (a:LegalArticle {id:$s}), (b:LegalArticle {id:$t}) "
          "CREATE (a)-[:DEFINED_BY]->(b)", {'s':src_id,'t':tgt_id})

# ESCALATES_TO: 심각도 에스컬레이션
escalation_edges = [
    ('level_4', 'level_5'),  # 반복 과속 시 형사처벌로 에스컬레이션
    ('level_5', 'level_6'),  # 극심한 과속
]
for src_lv, tgt_lv in escalation_edges:
    run_q("MATCH (s1:SeverityLevel {id:$s}), (s2:SeverityLevel {id:$t}) "
          "CREATE (s1)-[:ESCALATES_TO]->(s2)", {'s':src_lv,'t':tgt_lv})

# RELATED_TO: 교통안전법 ↔ DTG 관련 조문
safety_rels = [
    ('TSA_54_2', 'TSA_55_3'),  # DTG 장착 → 분석 결과 제공
    ('TSA_55_3', 'TSA_55_4'),  # 결과 제공 → 목적외 사용 금지
]
for s, t in safety_rels:
    run_q("MATCH (a:LegalArticle {id:$s}), (b:LegalArticle {id:$t}) "
          "CREATE (a)-[:RELATED_TO]->(b)", {'s':s,'t':t})

# PENALIZED_BY: 차종별 범칭금 연결
for pen in [('level_1','FINE_L1_FREIGHT'),('level_2','FINE_L2_FREIGHT'),
            ('level_3','FINE_L3_FREIGHT'),('level_4','FINE_L4_FREIGHT')]:
    run_q("MATCH (s:SeverityLevel {id:$lv}), (p:Penalty {id:$pid}) "
          "CREATE (s)-[:PENALIZED_BY]->(p)", {'lv':pen[0],'pid':pen[1]})

v3_edges = len(defined_by_edges) + len(escalation_edges) + len(safety_rels) + 4
print(f'✅ v3 확장 관계 추가: {v3_edges}개')


✅ VIOLATES: 31개
✅ CLASSIFIED_AS: 12개
✅ RELATED_TO: 12개
✅ PENALIZED_BY: 12개
✅ v3 확장 관계 추가: 11개


## 13. KG 통계

In [6]:
print('=== KG 통계 ===')
for label in ['HazardousBehavior','LegalArticle','SeverityLevel','Penalty','AggravatingFactor']:
    c = run_q(f'MATCH (n:{label}) RETURN count(n) AS c')[0]['c']
    print(f'  {label:25s}: {c}개')
print()
for rel in ['VIOLATES','CLASSIFIED_AS','RELATED_TO','PENALIZED_BY',
            'DEFINED_BY','ESCALATES_TO','AGGRAVATES']:
    c = run_q(f'MATCH ()-[r:{rel}]->() RETURN count(r) AS c')[0]['c']
    print(f'  -{rel:20s}: {c}개')

tot_n = run_q('MATCH (n) RETURN count(n) AS c')[0]['c']
tot_e = run_q('MATCH ()-[r]->() RETURN count(r) AS c')[0]['c']
print(f'\n총합: {tot_n}개 노드, {tot_e}개 엣지')

=== KG 통계 ===
  HazardousBehavior        : 13개
  LegalArticle             : 27개
  SeverityLevel            : 6개
  Penalty                  : 17개
  AggravatingFactor        : 2개

  -VIOLATES            : 31개
  -CLASSIFIED_AS       : 12개
  -RELATED_TO          : 14개
  -PENALIZED_BY        : 16개
  -DEFINED_BY          : 3개
  -ESCALATES_TO        : 2개


Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `AGGRAVATES` does not exist in database `1f2552f8`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=13, offset=12>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 12, 'line': 1, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH ()-[r:AGGRAVATES]->() RETURN count(r) AS c'


  -AGGRAVATES          : 0개

총합: 65개 노드, 78개 엣지


## 14. ★ KG 정확성 검증 (3종)
**검증 1**: SeverityLevel 노드 ↔ SGT(시행령 별표8) 대조
**검증 2**: RELATED_TO 엣지 ↔ 법조문 실제 참조 관계
**검증 3**: 전수 경로 테스트 — 초과속도 입력 → 올바른 심각도+벌칙 반환

In [7]:
validation_results = {'tests': [], 'pass': 0, 'fail': 0, 'total': 0}

def record(test_name, passed, detail=''):
    validation_results['total'] += 1
    if passed:
        validation_results['pass'] += 1
        print(f'  ✅ {test_name}: {detail}')
    else:
        validation_results['fail'] += 1
        print(f'  ❌ {test_name}: {detail}')
    validation_results['tests'].append({
        'test': test_name, 'passed': passed, 'detail': detail
    })

# ═══════════════════════════════════════
# 검증 1: SeverityLevel 노드 ↔ SGT 대조
# ═══════════════════════════════════════
print('=== 검증 1: SeverityLevel ↔ SGT (시행령 별표8) ===')

kg_levels = {r['id']: r for r in run_q("""
    MATCH (s:SeverityLevel)
    RETURN s.id AS id, s.label AS label, s.speed_over_min AS smin,
           s.speed_over_max AS smax, s.criminal AS criminal,
           s.legal_basis AS basis, s.demerit_points AS demerit
    ORDER BY s.id
""")}

for lv_id, sgt in SEVERITY_TABLE.items():
    kg = kg_levels.get(lv_id)
    if not kg:
        record(f'{lv_id} 존재', False, 'KG에 노드 없음')
        continue

    record(f'{lv_id} 존재', True, f'{kg["label"]}')

    # 초과속도 범위
    sgt_min = sgt['speed_over_min']
    sgt_max = sgt['speed_over_max']
    record(f'{lv_id} 속도범위',
           kg['smin'] == sgt_min and kg['smax'] == sgt_max,
           f'KG:[{kg["smin"]},{kg["smax"]}] vs SGT:[{sgt_min},{sgt_max}]')

    # 형사/행정
    record(f'{lv_id} 형사여부',
           kg['criminal'] == sgt['criminal'],
           f'KG:{kg["criminal"]} vs SGT:{sgt["criminal"]}')

    # 법적근거
    record(f'{lv_id} 법적근거',
           kg['basis'] == sgt['legal_basis'],
           f'KG:{kg["basis"]} vs SGT:{sgt["legal_basis"]}')

    # 벌점
    sgt_demerit = sgt.get('demerit_points')
    kg_demerit = kg['demerit'] if kg['demerit'] != -1 else None
    record(f'{lv_id} 벌점',
           kg_demerit == sgt_demerit,
           f'KG:{kg_demerit} vs SGT:{sgt_demerit}')

# ═══════════════════════════════════════
# 검증 2: RELATED_TO 엣지 정확성
# ═══════════════════════════════════════
print('\n=== 검증 2: RELATED_TO 엣지 ↔ 법조문 참조 관계 ===')

# 법적으로 올바른 참조 관계 (Ground Truth)
GROUND_TRUTH_RELATIONS = {
    # (source, target): 법적 근거
    ('RTA_17', 'RTA_156'): '제17조 속도위반 → 제156조 벌칙(20만원이하)',
    ('RTA_17', 'RTA_154'): '제17조 속도위반 → 제154조 벌칙(30만원이하, 80km/h초과)',
    ('RTA_17', 'RTA_153'): '제17조 속도위반 → 제153조 벌칙(100만원이하, 100km/h초과)',
    ('RTA_17', 'RTA_151_2'): '제17조 속도위반 → 제151조의2 반복초과속(3회이상)',
    ('RTA_49', 'RTA_156'): '제49조 준수사항위반(급감속 등) → 제156조 벌칙',
    ('RTA_19', 'RTA_156'): '제19조 안전거리위반 → 제156조 벌칙',
    ('RTA_21', 'RTA_156'): '제21조 앞지르기위반 → 제156조 벌칙',
    ('RTA_25', 'RTA_156'): '제25조 교차로통행위반 → 제156조 벌칙',
    ('RTA_12', 'RTA_17'): '제12조 어린이보호구역 → 제17조 속도위반 가중',
    ('TSA_55', 'TTBA_11'): '교통안전법 제55조 DTG 미제출 → 화물차법 제11조 행정처분',
    ('TSA_54_2', 'TSA_55'): '교통안전법 제54조의2 안전담당자 → 제55조 DTG관리',
    ('TTBA_59', 'TTBA_11'): '화물차법 제59조 교육미이수 → 제11조 행정처분',
    # ★ v3 추가 관계
    ('TSA_54_2', 'TSA_55_3'): '교통안전법 제54조의2 DTG장착 → 제55조③ 분석결과 제공',
    ('TSA_55_3', 'TSA_55_4'): '교통안전법 제55조③ 결과제공 → 제55조④ 목적외 사용 금지',
}

# KG에서 실제 RELATED_TO 엣지 조회
kg_relations = run_q("""
    MATCH (a1:LegalArticle)-[r:RELATED_TO]->(a2:LegalArticle)
    RETURN a1.id AS src, a2.id AS tgt, r.description AS desc
""")
kg_rel_set = {(r['src'], r['tgt']) for r in kg_relations}

# GT에 있는 관계가 KG에도 있는지
for (src, tgt), reason in GROUND_TRUTH_RELATIONS.items():
    record(f'RELATED {src}→{tgt}',
           (src, tgt) in kg_rel_set,
           reason)

# KG에 있지만 GT에 없는 관계 (잘못된 엣지)
spurious = kg_rel_set - set(GROUND_TRUTH_RELATIONS.keys())
if spurious:
    for s, t in spurious:
        record(f'불필요 엣지 {s}→{t}', False, 'GT에 없는 관계')
else:
    record('불필요 엣지 없음', True, f'KG 엣지 {len(kg_rel_set)}개 = GT {len(GROUND_TRUTH_RELATIONS)}개')

# ═══════════════════════════════════════
# 검증 3: 전수 경로 테스트 (초과속도 → 심각도 → 벌칙)
# ═══════════════════════════════════════
print('\n=== 검증 3: 전수 경로 테스트 (초과속도 → 심각도 → 벌칙) ===')

# 테스트 케이스: (초과속도, 기대 심각도, 기대 벌칙 키워드)
PATH_TEST_CASES = [
    (5,   'level_1', False, ['3만원']),
    (10,  'level_1', False, ['3만원']),
    (15,  'level_1', False, ['3만원']),
    (20,  'level_1', False, ['3만원']),
    (25,  'level_2', False, ['6만원', '15점']),
    (35,  'level_2', False, ['6만원', '15점']),
    (40,  'level_2', False, ['6만원', '15점']),
    (45,  'level_3', False, ['9만원', '30점']),
    (55,  'level_3', False, ['9만원', '30점']),
    (60,  'level_3', False, ['9만원', '30점']),
    (65,  'level_4', False, ['12만원', '60점']),
    (75,  'level_4', False, ['12만원', '60점']),
    (80,  'level_4', False, ['12만원', '60점']),
    (85,  'level_5', True,  ['30만원이하', '80점']),
    (95,  'level_5', True,  ['30만원이하', '80점']),
    (100, 'level_5', True,  ['30만원이하', '80점']),
    (105, 'level_6', True,  ['100만원이하', '면허취소']),
    (120, 'level_6', True,  ['100만원이하', '면허취소']),
    (150, 'level_6', True,  ['100만원이하', '면허취소']),
]

for speed_over, expected_lv, expected_criminal, expected_penalties in PATH_TEST_CASES:
    # KG Cypher 쿼리: 초과속도 → 심각도 → 벌칙
    result = run_q("""
        MATCH (s:SeverityLevel)
        WHERE s.speed_over_min < $so AND s.speed_over_max >= $so
        OPTIONAL MATCH (s)-[:PENALIZED_BY]->(p:Penalty)
        RETURN s.id AS lv, s.criminal AS criminal, collect(p.value) AS penalties
    """, {'so': speed_over})

    if not result:
        record(f'경로 {speed_over}km/h', False, 'KG에서 매칭되는 SeverityLevel 없음')
        continue

    r = result[0]
    lv_ok = r['lv'] == expected_lv
    crim_ok = r['criminal'] == expected_criminal
    pen_ok = all(any(ep in str(p) for p in r['penalties']) for ep in expected_penalties)

    all_ok = lv_ok and crim_ok and pen_ok
    detail = (f'초과 {speed_over}km/h → {r["lv"]}({"✓" if lv_ok else "✗"}) '
              f'형사:{r["criminal"]}({"✓" if crim_ok else "✗"}) '
              f'벌칙:{r["penalties"]}({"✓" if pen_ok else "✗"})')
    record(f'경로 {speed_over:3d}km/h', all_ok, detail)

# ═══════════════════════════════════════
# 검증 4: VIOLATES 엣지 — 위반유형→조문 매핑 정확성
# ═══════════════════════════════════════
print('\n=== 검증 4: VIOLATES 엣지 (위반유형→조문 매핑) ===')

for vtype, info in VIOLATION_MAPPING.items():
    expected_arts = set(
        info.get('definition_articles', []) +
        info.get('penalty_articles', []) +
        info.get('aggravation', [])
    )

    kg_arts = {r['aid'] for r in run_q("""
        MATCH (h:HazardousBehavior {name:$v})-[:VIOLATES]->(a:LegalArticle)
        RETURN a.id AS aid
    """, {'v': vtype})}

    missing = expected_arts - kg_arts
    extra = kg_arts - expected_arts

    all_ok = len(missing) == 0 and len(extra) == 0
    detail = f'기대 {len(expected_arts)}개, KG {len(kg_arts)}개'
    if missing: detail += f', 누락: {missing}'
    if extra: detail += f', 초과: {extra}'
    record(f'VIOLATES {vtype}', all_ok, detail)

# ═══════════════════════════════════════
# 검증 5: 경계값 정확성 (20, 40, 60, 80, 100km/h)
# ═══════════════════════════════════════
print('\n=== 검증 5: 경계값 정확성 (이하/초과 구분) ===')

# SGT 기준: speed_over_min < speed_over <= speed_over_max
# 즉 20km/h "이하" = level_1, 20km/h "초과" = level_2
BOUNDARY_TESTS = [
    (20,  'level_1', '20km/h = level_1 (20km/h 이하 초과)'),
    (21,  'level_2', '21km/h = level_2 (20km/h 초과)'),
    (40,  'level_2', '40km/h = level_2 (40km/h 이하 초과)'),
    (41,  'level_3', '41km/h = level_3 (40km/h 초과)'),
    (60,  'level_3', '60km/h = level_3 (60km/h 이하 초과)'),
    (61,  'level_4', '61km/h = level_4 (60km/h 초과)'),
    (80,  'level_4', '80km/h = level_4 (80km/h 이하 초과)'),
    (81,  'level_5', '81km/h = level_5 (80km/h 초과)'),
    (100, 'level_5', '100km/h = level_5 (100km/h 이하 초과)'),
    (101, 'level_6', '101km/h = level_6 (100km/h 초과)'),
]

for so, expected_lv, desc in BOUNDARY_TESTS:
    result = run_q("""
        MATCH (s:SeverityLevel)
        WHERE s.speed_over_min < $so AND s.speed_over_max >= $so
        RETURN s.id AS lv
    """, {'so': so})

    if result:
        actual_lv = result[0]['lv']
        record(f'경계 {so:3d}km/h', actual_lv == expected_lv,
               f'{desc} → KG: {actual_lv} {"✓" if actual_lv==expected_lv else "✗ 기대:"+expected_lv}')
    else:
        record(f'경계 {so:3d}km/h', False, f'{desc} → KG: 매칭 없음')

=== 검증 1: SeverityLevel ↔ SGT (시행령 별표8) ===
  ✅ level_1 존재: 경미
  ✅ level_1 속도범위: KG:[0,20] vs SGT:[0,20]
  ✅ level_1 형사여부: KG:False vs SGT:False
  ✅ level_1 법적근거: KG:제156조제1호 vs SGT:제156조제1호
  ✅ level_1 벌점: KG:0 vs SGT:0
  ✅ level_2 존재: 주의
  ✅ level_2 속도범위: KG:[20,40] vs SGT:[20,40]
  ✅ level_2 형사여부: KG:False vs SGT:False
  ✅ level_2 법적근거: KG:제156조제1호 vs SGT:제156조제1호
  ✅ level_2 벌점: KG:15 vs SGT:15
  ✅ level_3 존재: 경고
  ✅ level_3 속도범위: KG:[40,60] vs SGT:[40,60]
  ✅ level_3 형사여부: KG:False vs SGT:False
  ✅ level_3 법적근거: KG:제156조제1호 vs SGT:제156조제1호
  ✅ level_3 벌점: KG:30 vs SGT:30
  ✅ level_4 존재: 위험
  ✅ level_4 속도범위: KG:[60,80] vs SGT:[60,80]
  ✅ level_4 형사여부: KG:False vs SGT:False
  ✅ level_4 법적근거: KG:제156조제1호 vs SGT:제156조제1호
  ✅ level_4 벌점: KG:60 vs SGT:60
  ✅ level_5 존재: 매우위험
  ✅ level_5 속도범위: KG:[80,100] vs SGT:[80,100]
  ✅ level_5 형사여부: KG:True vs SGT:True
  ✅ level_5 법적근거: KG:제154조제9호 vs SGT:제154조제9호
  ✅ level_5 벌점: KG:80 vs SGT:80
  ✅ level_6 존재: 극히위험
  ✅ level_6 속도범위: KG:[100,9999] 

## 15. KG 검증 결과 요약

In [8]:
print('\n' + '='*60)
print('  KG 정확성 검증 결과')
print('='*60)
print(f'  총 테스트: {validation_results["total"]}건')
print(f'  통과:     {validation_results["pass"]}건 ✅')
print(f'  실패:     {validation_results["fail"]}건 ❌')
accuracy = validation_results['pass'] / max(validation_results['total'], 1)
print(f'  정확도:   {accuracy:.1%}')

if validation_results['fail'] > 0:
    print(f'\n  [실패 항목]')
    for t in validation_results['tests']:
        if not t['passed']:
            print(f'    ❌ {t["test"]}: {t["detail"]}')

# 논문용 보고
print(f'\n  [논문 보고]')
print(f'  "구축된 KG의 정확성을 {validation_results["total"]}건의 자동화된')
print(f'   검증 테스트로 확인하였으며, 전체 정확도는 {accuracy:.1%}이다."')

if accuracy == 1.0:
    print(f'\n  ✅ KG 정확성 검증 통과 — 실험 진행 가능')
else:
    print(f'\n  ⚠️ KG 정확성 {accuracy:.1%} — 실패 항목 수정 후 재검증 필요')

# 결과 저장
validation_results['accuracy'] = round(accuracy, 4)
save_json(validation_results, RESULTS_DIR / 'evaluation' / 'kg_validation.json')


  KG 정확성 검증 결과
  총 테스트: 87건
  통과:     87건 ✅
  실패:     0건 ❌
  정확도:   100.0%

  [논문 보고]
  "구축된 KG의 정확성을 87건의 자동화된
   검증 테스트로 확인하였으며, 전체 정확도는 100.0%이다."

  ✅ KG 정확성 검증 통과 — 실험 진행 가능
✅ 저장: results\evaluation\kg_validation.json


## 16. KG 스냅샷 저장

In [9]:
nodes = run_q('MATCH (n) RETURN labels(n)[0] AS label, properties(n) AS props')
edges = run_q('MATCH (a)-[r]->(b) RETURN a.id AS src, type(r) AS rel, b.id AS tgt, properties(r) AS props')

save_json({'nodes': nodes, 'edges': edges}, KG_DIR / 'kg_snapshot.json')
print(f'   노드: {len(nodes)}개, 엣지: {len(edges)}개')
print(f'\n→ 다음: 04_irac_v_framework.ipynb')

✅ 저장: data\kg\kg_snapshot.json
   노드: 65개, 엣지: 78개

→ 다음: 04_irac_v_framework.ipynb


## 15. ★ RAG용 VectorDB 구축 (FAISS + OpenAI Embedding)
**목적**: S2(Flat RAG)에서 사용할 법령 시맨틱 검색 인덱스

**임베딩 모델**: `text-embedding-3-small` (OpenAI, 1536dim)

**코퍼스**: 법령 조문 원문 + 시행령 별표(벌점·범칙금)

**산출물**: `data/kg/faiss_index.bin`, `data/kg/rag_chunks.json`, `data/kg/rag_metadata.json`

In [10]:
# pip install faiss-cpu openai

import numpy as np
import faiss
from openai import OpenAI

oai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
EMBED_MODEL = 'text-embedding-3-small'  # 1536 dim, 최신 경량 모델

# ── 1) 청킹: 법령 원문 + 별표 ──
rag_chunks = []
rag_meta = []

for art in CORE_ARTICLES:
    content = art.get('content', '')
    if len(content) < 20:
        continue
    # 300자 단위, 100자 overlap
    for i_c in range(0, len(content), 200):
        chunk = content[i_c:i_c+300]
        if len(chunk) >= 20:
            rag_chunks.append(chunk)
            rag_meta.append({'id':art['id'], 'law':art.get('law',''),
                             'jo_num':art.get('jo_num',''), 'title':art.get('title','')})

# 시행령 별표 (벌점·범칙금 기준)
penalty_texts = [
    ('RTAE_T8', '도로교통법 시행령', '별표 8 범칙금액',
     '과속 범칙금: 승합·화물 20km/h이하 3만원, 20~40km/h 7만원, '
     '40~60km/h 10만원, 60~80km/h 13만원'),
    ('RTAE_T7', '도로교통법 시행령', '별표 7 벌점기준',
     '과속 벌점: 20km/h이하 없음, 20~40km/h 15점, '
     '40~60km/h 30점, 60~80km/h 60점, 80~100km/h 80점'),
    ('RTAE_T10', '도로교통법 시행령', '별표 10 보호구역',
     '어린이보호구역 과속 범칭금 가중 적용'),
    ('RTA_154', '도로교통법', '제154조 벌칙',
     '80km/h 초과 과속: 30만원 이하 벌금 또는 구류 (형사처벌)'),
    ('RTA_153', '도로교통법', '제153조 벌칙',
     '100km/h 초과 과속: 100만원 이하 벌금 또는 구류 (형사처벌)'),
]
for pid, law, title, content in penalty_texts:
    rag_chunks.append(content)
    rag_meta.append({'id':pid, 'law':law, 'jo_num':title, 'title':title})

print(f'✅ RAG 청크: {len(rag_chunks)}건')

# ── 2) OpenAI 임베딩 생성 ──
def get_embeddings(texts, model=EMBED_MODEL, batch_size=100):
    """OpenAI 임베딩 API 배치 호출."""
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        resp = oai_client.embeddings.create(input=batch, model=model)
        all_embs.extend([d.embedding for d in resp.data])
        if i > 0: print(f'  임베딩 진행: {i+len(batch)}/{len(texts)}')
    return np.array(all_embs, dtype='float32')

print(f'임베딩 생성 중 ({EMBED_MODEL})...')
chunk_embeddings = get_embeddings(rag_chunks)
print(f'✅ 임베딩 완료: shape={chunk_embeddings.shape}')

# ── 3) FAISS 인덱스 구축 ──
dim = chunk_embeddings.shape[1]
# L2 정규화 후 Inner Product = Cosine Similarity
faiss.normalize_L2(chunk_embeddings)
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(chunk_embeddings)
print(f'✅ FAISS 인덱스: {faiss_index.ntotal}건, dim={dim}')

# ── 4) 디스크 저장 ──
faiss.write_index(faiss_index, str(KG_DIR / 'faiss_index.bin'))
save_json(rag_chunks, KG_DIR / 'rag_chunks.json')
save_json(rag_meta, KG_DIR / 'rag_metadata.json')
print(f'\n✅ 저장 완료:')
print(f'  {KG_DIR}/faiss_index.bin ({faiss_index.ntotal} vectors, {dim}d)')
print(f'  {KG_DIR}/rag_chunks.json ({len(rag_chunks)}건)')
print(f'  {KG_DIR}/rag_metadata.json ({len(rag_meta)}건)')


✅ RAG 청크: 46건
임베딩 생성 중 (text-embedding-3-small)...
✅ 임베딩 완료: shape=(46, 1536)
✅ FAISS 인덱스: 46건, dim=1536
✅ 저장: data\kg\rag_chunks.json
✅ 저장: data\kg\rag_metadata.json

✅ 저장 완료:
  data\kg/faiss_index.bin (46 vectors, 1536d)
  data\kg/rag_chunks.json (46건)
  data\kg/rag_metadata.json (46건)


In [12]:
# ── 검색 테스트 ──
def _test_search(query, top_k=3):
    q_emb = get_embeddings([query])
    faiss.normalize_L2(q_emb)
    scores, indices = faiss_index.search(q_emb, top_k)
    print(f'\nQuery: "{query}"')
    for idx, score in zip(indices[0], scores[0]):
        if idx < 0: continue
        print(f'  [{rag_meta[idx]["id"]}] score={score:.3f}: {rag_chunks[idx][:60]}...')

_test_search('화물차 과속 벌칙 범칙금')
_test_search('어린이보호구역 속도위반')
_test_search('급감속 급정지 안전운전 의무')

print(f'\n→ 다음: 04_irac_v_framework.ipynb (FAISS 인덱스 로드하여 S2 검색에 사용)')



Query: "화물차 과속 벌칙 범칙금"
  [RTAE_T8] score=0.555: 과속 범칙금: 승합·화물 20km/h이하 3만원, 20~40km/h 7만원, 40~60km/h 10만원, 6...
  [RTA_153] score=0.480: 100km/h 초과 과속: 100만원 이하 벌금 또는 구류 (형사처벌)...
  [TTBA_11] score=0.480: 자동차를 소유한 위ㆍ수탁차주나 개인 운송사업자에게 화물운송을 위탁하는 경우 국토교통부령으로 정하는 화물을 제...

Query: "어린이보호구역 속도위반"
  [RTAE_T10] score=0.727: 어린이보호구역 과속 범칭금 가중 적용...
  [RTA_12] score=0.543: ① 시장등은 교통사고의 위험으로부터 어린이를 보호하기 위하여 필요하다고 인정하는 경우에는 초등학교, 유치원,...
  [RTA_17] score=0.403: 우에는 다음 각 호의 구분에 따라 구역이나 구간을 지정하여 제1항에 따라 정한 속도를 제한할 수 있다. <개...

Query: "급감속 급정지 안전운전 의무"
  [TTBA_11] score=0.471:  <신설 2018.8.14> ㉓ 운송사업자는 「자동차관리법」 제35조를 위반하여 전기ㆍ전자장치(최고속도제한장...
  [RTA_17] score=0.459: ① 자동차등(개인형 이동장치는 제외한다. 이하 이 조에서 같다)과 노면전차의 도로 통행 속도는 행정안전부령으...
  [RTA_25] score=0.459: 에 있는 차 또는 노면전차의 상황에 따라 교차로(정지선이 설치되어 있는 경우에는 그 정지선을 넘은 부분을 말...

→ 다음: 04_irac_v_framework.ipynb (FAISS 인덱스 로드하여 S2 검색에 사용)
